# Multi-LoRA Serving with vLLM

Trains 3 LoRA adapters (SQL generation, dialogue summarization, NER->JSON extraction) on top of one base model, then serves all 3 simultaneously from a single vLLM engine instance.

**Before running:** push the project repo to GitHub and set `REPO_URL` below. Also add your Hugging Face token as a Kaggle secret named `HF_TOKEN` (Add-ons -> Secrets) since the default base model is gated.

In [ ]:
REPO_URL = "https://github.com/<your-username>/vllm-multi-lora-serving.git"
BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"  # swap for Qwen/Qwen2.5-3B-Instruct to skip HF gating

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
!git clone {REPO_URL} repo
%cd repo
!pip install -q -r requirements.txt

## 1. Prepare the 3 instruction datasets

In [ ]:
!python src/prepare_data.py --output-dir data --n-train 2000 --n-test 20

## 2. Train each LoRA adapter

Run sequentially — each is a separate fine-tune on top of the same frozen base model. Roughly 20-40 min per task on a T4.

In [ ]:
!python src/train_lora.py --task sql --base-model {BASE_MODEL} --data-dir data --output-dir adapters

In [ ]:
!python src/train_lora.py --task summarize --base-model {BASE_MODEL} --data-dir data --output-dir adapters

In [ ]:
!python src/train_lora.py --task extract --base-model {BASE_MODEL} --data-dir data --output-dir adapters

## 3. Install vLLM and serve all 3 adapters from one engine

Installed separately since vLLM pins its own torch/cuda build.

In [ ]:
!pip install -q vllm

In [ ]:
!python src/serve_demo.py --base-model {BASE_MODEL} --adapters-dir adapters --mode compare

## 4. (Optional) Save trained adapters as a Kaggle output/dataset

The `adapters/` directory now holds `sql-lora/`, `summarize-lora/`, and `extract-lora/` — small enough to persist as Kaggle Notebook Output or a new Kaggle Dataset for reuse without retraining.